# 03 · Walk-forward validation

Does the XGBoost gain survive when the test year moves? Two expanding-window folds.

**Key results**
- Fold A (train → 2018, test 2019): XGBoost 7.50 vs naive 8.51 MU (+12%).
- Fold B (train → 2020, test 2021): XGBoost 4.49 vs naive 5.35 MU (+16%).
- Only 148/253 (2019) and 15/96 (2021) origins have a complete 56-day history — long-context models will see far fewer usable windows on this dataset.

> Paths are relative to `notebooks/`. Outputs are from the original research run; see [`data/README.md`](../data/README.md) for data that must be downloaded.

➡ Next: [04 · Zero-shot Chronos-Bolt](04_zero_shot_chronos_bolt.ipynb)

Prepare Fold A

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("../data")

features = pd.read_parquet(
    DATA / "processed/gridpulse_features_v1.parquet"
)

EXCLUDE = {"origin", "target_date", "target", "split"}

FEATURE_COLS = [
    c for c in features.columns
    if c not in EXCLUDE
]

# Entire seven-day target window must belong
# to the appropriate chronological period.
bounds = features.groupby("origin")["target_date"].agg(
    first="min",
    last="max"
)

train_origins = bounds.index[
    bounds["last"] < pd.Timestamp("2019-01-01")
]

val_origins = bounds.index[
    (bounds["first"] >= pd.Timestamp("2019-01-01"))
    & (bounds["last"] < pd.Timestamp("2020-01-01"))
]

train_a = features[
    features["origin"].isin(train_origins)
].copy()

val_a = features[
    features["origin"].isin(val_origins)
].copy()

print("Training origins:", train_a["origin"].nunique())
print("Validation origins:", val_a["origin"].nunique())
print("Training rows:", len(train_a))
print("Validation rows:", len(val_a))

assert train_a["target_date"].max() < val_a["target_date"].min()
assert len(train_a) % 7 == 0
assert len(val_a) % 7 == 0

Training origins: 963
Validation origins: 253
Training rows: 6741
Validation rows: 1771


Train and evaluate Fold A

In [2]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error

X_train_a = train_a[FEATURE_COLS]
y_train_a = train_a["target"]

X_val_a = val_a[FEATURE_COLS]
y_val_a = val_a["target"]

model_a = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    device="cpu",
    random_state=42,
    n_jobs=-1,
)

model_a.fit(X_train_a, y_train_a)

pred_a = model_a.predict(X_val_a)

mae_a = mean_absolute_error(y_val_a, pred_a)
rmse_a = np.sqrt(mean_squared_error(y_val_a, pred_a))
wape_a = (
    100 * np.abs(y_val_a - pred_a).sum()
    / np.abs(y_val_a).sum()
)

print(f"Fold A XGBoost MAE:  {mae_a:.3f} MU")
print(f"Fold A XGBoost RMSE: {rmse_a:.3f} MU")
print(f"Fold A XGBoost WAPE: {wape_a:.2f}%")

Fold A XGBoost MAE:  7.495 MU
Fold A XGBoost RMSE: 10.048 MU
Fold A XGBoost WAPE: 8.05%


Evaluate weekly naive on the identical 2019 windows

In [3]:
naive_pred_a = np.array([
    row[f"lag_{8 - int(row['horizon'])}"]
    for _, row in val_a.iterrows()
])

naive_mae_a = mean_absolute_error(y_val_a, naive_pred_a)
naive_rmse_a = np.sqrt(
    mean_squared_error(y_val_a, naive_pred_a)
)

improvement_a = 100 * (
    naive_mae_a - mae_a
) / naive_mae_a

print(f"Fold A weekly naive MAE: {naive_mae_a:.3f} MU")
print(f"Fold A XGBoost MAE:      {mae_a:.3f} MU")
print(f"XGBoost improvement:     {improvement_a:+.2f}%")

Fold A weekly naive MAE: 8.512 MU
Fold A XGBoost MAE:      7.495 MU
XGBoost improvement:     +11.95%


Preserve Fold A predictions

In [4]:
fold_a_results = val_a[
    ["origin", "target_date", "horizon", "target"]
].copy()

fold_a_results["xgb_pred"] = pred_a
fold_a_results["naive_pred"] = naive_pred_a

fold_a_results.to_parquet(
    DATA / "processed/walkforward_fold_a.parquet",
    index=False,
)

Fold B: Train through 2020, evaluate on 2021

In [5]:
# Fold B: chronological boundaries
train_origins_b = bounds.index[
    bounds["last"] < pd.Timestamp("2021-01-01")
]

val_origins_b = bounds.index[
    (bounds["first"] >= pd.Timestamp("2021-01-01"))
    & (bounds["last"] < pd.Timestamp("2022-01-01"))
]

train_b = features[
    features["origin"].isin(train_origins_b)
].copy()

val_b = features[
    features["origin"].isin(val_origins_b)
].copy()

print("Training origins:", train_b["origin"].nunique())
print("Validation origins:", val_b["origin"].nunique())
print("Training rows:", len(train_b))
print("Validation rows:", len(val_b))

assert train_b["target_date"].max() < val_b["target_date"].min()
assert len(train_b) % 7 == 0
assert len(val_b) % 7 == 0

Training origins: 1345
Validation origins: 96
Training rows: 9415
Validation rows: 672


In [6]:
model_b = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    tree_method="hist",
    device="cpu",
    random_state=42,
    n_jobs=-1,
)

model_b.fit(train_b[FEATURE_COLS], train_b["target"])

pred_b = model_b.predict(val_b[FEATURE_COLS])

naive_pred_b = np.array([
    row[f"lag_{8 - int(row['horizon'])}"]
    for _, row in val_b.iterrows()
])

actual_b = val_b["target"].to_numpy()

xgb_mae_b = np.abs(actual_b - pred_b).mean()
naive_mae_b = np.abs(actual_b - naive_pred_b).mean()

print(f"Fold B weekly naive MAE: {naive_mae_b:.3f} MU")
print(f"Fold B XGBoost MAE:      {xgb_mae_b:.3f} MU")
print(
    f"XGBoost improvement:     "
    f"{100 * (naive_mae_b - xgb_mae_b) / naive_mae_b:+.2f}%"
)

fold_b_results = val_b[
    ["origin", "target_date", "horizon", "target"]
].copy()

fold_b_results["xgb_pred"] = pred_b
fold_b_results["naive_pred"] = naive_pred_b

fold_b_results.to_parquet(
    DATA / "processed/walkforward_fold_b.parquet",
    index=False,
)

Fold B weekly naive MAE: 5.353 MU
Fold B XGBoost MAE:      4.490 MU
XGBoost improvement:     +16.12%


Check 56-day eligibility across folds

In [7]:
energy = pd.read_csv(
    DATA / "processed/delhi_energy_clean.csv",
    parse_dates=["Date"],
    index_col="Date",
).sort_index()["energy_met_MU"]

fold_files = {
    "2019": "walkforward_fold_a.parquet",
    "2021": "walkforward_fold_b.parquet",
}

eligible_origins = {}

for year, filename in fold_files.items():
    predictions = pd.read_parquet(
        DATA / "processed" / filename
    )

    origins = predictions["origin"].drop_duplicates().sort_values()

    valid = []

    for origin in origins:
        context = energy.loc[
            origin - pd.Timedelta(days=55):origin
        ]

        if len(context) == 56 and context.notna().all():
            valid.append(origin)

    eligible_origins[year] = valid

    print(
        f"{year}: {len(valid)} / {len(origins)} "
        "origins support 56-day context"
    )

2019: 148 / 253 origins support 56-day context
2021: 15 / 96 origins support 56-day context


Compare 28-day and 56-day eligibility

In [8]:
for year, filename in fold_files.items():
    predictions = pd.read_parquet(
        DATA / "processed" / filename
    )

    origins = predictions["origin"].drop_duplicates().sort_values()

    print(f"\n{year}")

    for length in [28, 42, 56]:
        eligible = 0

        for origin in origins:
            history = energy.loc[
                origin - pd.Timedelta(days=length - 1):origin
            ]

            if len(history) == length and history.notna().all():
                eligible += 1

        print(f"{length}-day context: {eligible}/{len(origins)}")


2019
28-day context: 253/253
42-day context: 204/253
56-day context: 148/253

2021
28-day context: 96/96
42-day context: 42/96
56-day context: 15/96
